# Exemplo completo

In [ ]:
# ruff: noqa: T201
# Importando as bibliotecas

from anonimizar import Anonimizar, Evaluation, Trainer

## Atibuindo os objetos

trainer = Trainer(output_dir="./treinamento_ner")

## Gerando dados

data_list = [
    {"text": "Exemplo de texto com CPF 123.456.789-09.", "entities": [(25, 39, "CPF")]},
    {"text": "Outro exemplo de texto com CPF 123.456.789-09.", "entities": [(31, 45, "CPF")]},
    {"text": "Exemplo de texto com CPF 123.456.789-09.", "entities": [(25, 39, "CPF")]},
    {
        "text": "JoÃ£o Silva, CPF 123.456.789-09, mora na Rua das Flores, 123.",
        "entities": [(16, 30, "CPF"), (40, 59, "ENDEREÃ‡O")],
    },
    {
        "text": "Email de contato: joao@empresa.com.br, seu CPF Ã© 123.456.789-09",
        "entities": [(18, 37, "EMAIL"), (49, 63, "CPF")],
    },
    {
        "text": "Caio Silva, CPF 123.456.789-09, mora na Rua dos Flores, 123.",
        "entities": [(16, 30, "CPF"), (40, 59, "ENDEREÃ‡O")],
    },
    {
        "text": "Email de contato: caio@empresa.com.br, seu CPF Ã© 123.456.789-09",
        "entities": [(18, 37, "EMAIL"), (49, 63, "CPF")],
    },
    {"text": "Documento sem entidades para teste.", "entities": []},
]
trainer.add_data(data_list, errors="coerce")

## Treino
trainer.train(n_iter=30, drop=0.2, batch_size=4, validation_split=0.2)
trainer.save_model()
print("Treinamento concluÃ­do com sucesso!")
print(f"Total de exemplos de treinamento: {len(trainer.training_data)}")

anonymizer = Anonimizar("./treinamento_ner/")
anonymizer.add_apply_patterns(use_model_labels=True)

## Validacao

df_texts, df_gt = trainer.val_data_to_evaluation()

evaluator = Evaluation()

evaluator.load_data(df_texts, df_gt)

preds = evaluator.extract_predictions(anonymizer)
results = evaluator.evaluate_model()
print(evaluator.get_summary_report())

2025-07-25 15:02:37,661 - INFO - Inicializando Treinamento para o SeiAnonimizar com model_name: None
2025-07-25 15:02:37,662 - INFO - O modelo serÃ¡ salvo em: treinamento_ner
2025-07-25 15:02:37,768 - WARNING - Nao foram definidos labels, serÃ£o usados os labels padrÃ£o
2025-07-25 15:02:37,769 - INFO - Adicionado label CPF ao modelo
2025-07-25 15:02:37,773 - INFO - Adicionado label RG ao modelo
2025-07-25 15:02:37,777 - INFO - Adicionado label SIAPE ao modelo
2025-07-25 15:02:37,780 - INFO - Adicionado label ENDEREÃ‡O ao modelo
2025-07-25 15:02:37,784 - INFO - Adicionado label TELEFONE ao modelo
2025-07-25 15:02:37,787 - INFO - Adicionado label EMAIL ao modelo
2025-07-25 15:02:37,791 - INFO - Adicionado label FISTEL ao modelo
2025-07-25 15:02:37,794 - INFO - Adicionado label DADOS_BANCARIOS ao modelo
2025-07-25 15:02:37,798 - INFO - Adicionado label CNH ao modelo
2025-07-25 15:02:37,801 - INFO - Adicionado label PASSAPORTE ao modelo
2025-07-25 15:02:37,805 - INFO - Adicionado label TIT

Treinamento concluÃ­do com sucesso!
Total de exemplos de treinamento: 7


2025-07-25 15:02:41,840 - WARNING - Nao foi habilitado nenhum label, seram habilitados todos.
2025-07-25 15:02:41,841 - INFO - Modelo carregado com sucesso. Max length: 3000000
2025-07-25 15:02:41,843 - INFO - Labels configurados: {'CPF', 'ENDEREÃ‡O', 'SIAPE', 'TITULO_ELEITOR', 'CNH', 'EMAIL', 'RG', 'DATA_NASCIMENTO', 'PASSAPORTE', 'DADOS_BANCARIOS', 'TELEFONE'}
2025-07-25 15:02:41,845 - INFO - ValidaÃ§Ã£o de CPF ativa: True
2025-07-25 15:02:41,847 - INFO - Adicionando padrÃµes de CPF.
2025-07-25 15:02:41,849 - INFO - Adicionando padroes de EndereÃ§os.
2025-07-25 15:02:41,850 - INFO - Adicionando padrÃµes de SIAPE.
2025-07-25 15:02:41,851 - INFO - Adicionando padrÃµes de TITULO_ELEITOR.
2025-07-25 15:02:41,852 - INFO - Adicionando padrÃµes de CNH.
2025-07-25 15:02:41,853 - INFO - Adicionando padrÃµes de EMAIL.
2025-07-25 15:02:41,854 - INFO - Adicionando padrÃµes de RG.
2025-07-25 15:02:41,856 - INFO - Adicionando padrÃµes de DATA_NASCIMENTO.
2025-07-25 15:02:41,857 - INFO - Adicionand

=== RELATÃ“RIO DE AVALIAÃ‡ÃƒO ===

MÃ‰TRICAS CONSOLIDADAS:
  Entidades: 3
  IDs Ãºnicos: 2
  F-beta: 1.0000
  PrecisÃ£o: 1.0000
  Recall: 1.0000
  TP: 3, FP: 0, FN: 0

MÃ‰TRICAS POR TIPO DE ENTIDADE:

EMAIL:
  Entidades: 1
  F-beta: 1.0000
  PrecisÃ£o: 1.0000
  Recall: 1.0000

CPF:
  Entidades: 2
  F-beta: 1.0000
  PrecisÃ£o: 1.0000
  Recall: 1.0000


In [3]:
## Aplicando em novos casos
entidades = anonymizer.extract_entities(
    "Oi, meu nome Ã© Matheus, meu email Ã© exemplo@gmail.com, e meu CPF Ã© 123.456.789-09"
)
anonymizer.anonymize_text(
    "Oi, meu nome Ã© Matheus, meu email Ã© exemplo@gmail.com, e meu CPF Ã© 123.456.789-09", entidades
)

'Oi, meu nome Ã© Matheus, meu email Ã© <|EMAIL|>, e meu CPF Ã© <|CPF|>'

## Outros exemplos

In [ ]:
# ruff: noqa: T201
# 1) Carrega e seleciona uma amostra pequena
from pathlib import Path

import pandas as pd

df_textos = pd.read_parquet(Path("./textos.parquet"))
df_entidades = pd.read_parquet(Path("./entidades.parquet"))
if df_textos.empty:
    raise ValueError("O arquivo de textos nÃ£o pode estar vazio.")
df_texts_sample = df_textos.sample(n=min(20, len(df_textos)), random_state=42).reset_index(drop=True)
ids_keep = set(df_texts_sample["id"])
df_ents_sample = df_entidades[df_entidades["id"].isin(ids_keep)].copy()

# 2) Prepara trainer e dados
labels = [
    "CPF",
    "RG",
    "SIAPE",
    "ENDEREÃ‡O",
    "TELEFONE",
    "EMAIL",
    "DADOS_BANCARIOS",
    "CNH",
    "PASSAPORTE",
    "TITULO_ELEITOR",
    "DATA_NASCIMENTO",
    "GEO_COORD",
]
trainer = Trainer(labels=labels, output_dir="./demo_model")
trainer.add_data(df_ents_sample, errors="coerce")

# 3) Treina rÃ¡pido
trainer.train(n_iter=5, drop=0.3, batch_size=8, validation_split=0.2)
trainer.save_model("./demo_model")

# 4) Gera pares para avaliaÃ§Ã£o
df_texts_val, df_gt_val = trainer.val_data_to_evaluation()

# 5) Avalia
anonymizer = Anonimizar("./demo_model", labels=labels)
anonymizer.add_apply_patterns(labels=labels)

evaluator = Evaluation(overlap_threshold=0.8, beta=2.0)
evaluator.load_data(df_texts_val.rename(columns={"id": "id", "text": "text"}), df_gt_val)
preds = evaluator.extract_predictions(anonymizer)
results = evaluator.evaluate_model()
print(evaluator.get_summary_report())

# 6) Aplica em textos inÃ©ditos e anonimiza
novos_textos = [
    "Meu CPF Ã© 123.456.789-09 e meu e-mail Ã© teste@example.com.",
    "Contato: (11) 98765-4321. EndereÃ§o: Rua das Flores, 123 - CEP 01234-567.",
    "Passaporte BR123456, SIAPE 1234567, data de nascimento 15 de marÃ§o de 1985.",
]
for t in novos_textos:
    ents = anonymizer.extract_entities(t, return_type="label_detail")
    print("Original:", t)
    print("Entidades:", ents)
    print("Anonimizado:", anonymizer.anonymize_text(t, ents))
    print("-" * 80)